# 05 — Feed Pipeline
Explore → Extract → Analyse → Ingest → Verify

This notebook covers the **feed** object type from the raw Backyard dump.

**Scope (enforced in extractor):**
- Include only `feed_type in {post, recognition}`
- For `feed_type == post`, include only `feed_variant == standard`
- Exclude `timeline` feeds and post variants `shared_content` / `shared_recognition`

**Graph model:**

| Node | Key properties |
|------|----------------|
| `Post` | id, feed_type, feed_variant, feed_author_name, feed_ref_data, feed_site_name, feed_content_title |
| `Recognition` | id, feed_type, feed_variant, feed_author_name |

| Relationship | Pattern |
|---|---|
| `CREATED` | (People)→(Post\|Recognition) |
| `LAST_MODIFIED` | (People)→(Post\|Recognition) |
| `BELONGS_TO_SITE` | (Post)→(Site) |
| `BELONGS_TO_CONTENT` | (Post)→(Content) |
| `MENTIONS` | (Post)→(People\|Site\|Content) |
| `RECOGNIZES` | (Recognition)→(People) |

Assumes pipelines 01 (People), 02 (Site), 03 (File), and 04 (Content) are already loaded.

**Sections:**

0. Setup
1. Explore raw data
2. Extract & normalise
3. Analyse extraction results
4. Ingest into Neo4j
5. Post-ingestion verification

---
## 0 · Setup

In [1]:
import json
import sys
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import RAW_FILE, NORMALIZED_DIR, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD

ALLOWED_FEED_TYPES = {"post", "recognition"}
ALLOWED_POST_VARIANTS = {"standard"}

print(f"Project root      : {PROJECT_ROOT}")
print(f"Raw file          : {RAW_FILE}")
print(f"Raw file exists   : {RAW_FILE.exists()}")
print(f"Normalized dir    : {NORMALIZED_DIR}")
print(f"Normalized exists : {NORMALIZED_DIR.exists()}")

Project root      : /home/ubuntu/graph_experiments
Raw file          : /home/ubuntu/graph_experiments/data/raw/backyard_dump_19022026_172648.jsonl
Raw file exists   : True
Normalized dir    : /home/ubuntu/graph_experiments/data/normalized
Normalized exists : True


---
## 1 · Explore Raw Data

Load all feed records from the raw dump and apply scope filters.

In [2]:
# Load all feed records and cross-reference entity IDs
feed_items = []
people_ids = set()
site_ids = set()
content_ids = set()

with open(RAW_FILE) as f:
    for line in f:
        obj = json.loads(line)
        ot = obj.get("object_type")
        if ot == "feed":
            feed_items.append(obj)
        elif ot == "people" and obj.get("id"):
            people_ids.add(obj["id"])
        elif ot == "site" and obj.get("id"):
            site_ids.add(obj["id"])
        elif ot == "content" and obj.get("id"):
            content_ids.add(obj["id"])

print(f"Feed records (all)   : {len(feed_items)}")
print(f"People IDs (raw)     : {len(people_ids)}")
print(f"Site IDs (raw)       : {len(site_ids)}")
print(f"Content IDs (raw)    : {len(content_ids)}")

Feed records (all)   : 6029
People IDs (raw)     : 954
Site IDs (raw)       : 45
Content IDs (raw)    : 3142


In [3]:
# Save 10 sample feed records for debugging
sample_path = PROJECT_ROOT / "data" / "debug" / "sample_feeds.jsonl"
sample_path.parent.mkdir(parents=True, exist_ok=True)
with open(sample_path, "w") as out:
    for s in feed_items[:10]:
        out.write(json.dumps(s) + "\n")
print(f"Saved {min(10, len(feed_items))} samples → {sample_path}")

# Pretty-print first record
print("\n--- Sample record ---")
print(json.dumps(feed_items[0], indent=2))

Saved 10 samples → /home/ubuntu/graph_experiments/data/debug/sample_feeds.jsonl

--- Sample record ---
{
  "feed_variant": "standard",
  "feed_shared_entity_id": null,
  "feed_site_id": "a45f90cc-8d69-4b68-a076-277efce515db",
  "feed_content_id": "5754b18e-1b51-4e25-85b3-1f9135ef79ce",
  "createddate": "2025-01-07T11:31:25.256Z",
  "feed_author_image_url": "/content/o/b3ef5b36-5642-4e21-9701-48997cd4f1a5",
  "modifiedbyid": "7b19c93d-ab26-4074-8a9c-110ac260ffb7",
  "feed_content_title": "2026 Global Holiday & Recharge Days",
  "createdbyid": "7b19c93d-ab26-4074-8a9c-110ac260ffb7",
  "feed_site_is_active": true,
  "feed_deleted_files": [],
  "feed_site_name": "Life at Simpplr",
  "is_deleted": false,
  "feed_html_data": "<p>how to get 2024 global holiday list?</p>",
  "feed_author_name": "Biswajit Das",
  "feed_data": "{\"type\":\"doc\",\"content\":[{\"type\":\"paragraph\",\"attrs\":{\"className\":\"\",\"data-sw-sid\":null},\"content\":[{\"type\":\"text\",\"text\":\"how to get 2024 glob

In [4]:
# Scope filter — same logic as feed_extractor.keep_record()
def keep_for_scope(row):
    ft = row.get("feed_type")
    fv = row.get("feed_variant")
    if ft not in ALLOWED_FEED_TYPES:
        return False
    if ft == "post" and fv not in ALLOWED_POST_VARIANTS:
        return False
    return True

feeds = [r for r in feed_items if keep_for_scope(r)]
excluded = [r for r in feed_items if not keep_for_scope(r)]

excluded_by_type = Counter(str(r.get("feed_type")) for r in excluded)
excluded_post_by_variant = Counter(
    str(r.get("feed_variant")) for r in excluded if r.get("feed_type") == "post"
)

print(f"Rows retained after scope filter : {len(feeds)}")
print(f"Rows excluded                    : {len(excluded)}")
print(f"  Excluded by feed_type          : {dict(excluded_by_type)}")
print(f"  Excluded post variants         : {dict(excluded_post_by_variant)}")

Rows retained after scope filter : 3780
Rows excluded                    : 2249
  Excluded by feed_type          : {'timeline': 2132, 'post': 117}
  Excluded post variants         : {'shared_content': 61, 'shared_recognition': 41, 'shared_feed_post': 10, 'shared_social_campaign': 3, 'shared_award': 2}


### 1.1 Subtype and variant overview (scoped)

In [5]:
print("=== feed_type ===")
print(Counter(r.get("feed_type") for r in feeds).most_common())

print("\n=== feed_variant ===")
print(Counter(r.get("feed_variant") for r in feeds).most_common())

print("\n=== TYPE × VARIANT matrix (scoped) ===")
matrix = Counter((str(r.get("feed_type")), str(r.get("feed_variant"))) for r in feeds)
for (t, v), c in sorted(matrix.items()):
    print(f"  {t} | {v}: {c}")

=== feed_type ===
[('recognition', 2179), ('post', 1601)]

=== feed_variant ===
[('shared_recognition', 2179), ('standard', 1601)]

=== TYPE × VARIANT matrix (scoped) ===
  post | standard: 1601
  recognition | shared_recognition: 2179


### 1.2 Field coverage by feed_type

In [6]:
fields = [
    "id", "createdbyid", "modifiedbyid", "feed_author_name",
    "feed_data", "feed_html_data",
    "feed_site_id", "feed_site_name", "feed_site_type",
    "feed_content_id", "feed_content_type",
    "feed_ref_data", "feed_associated_entity",
    "feed_files", "feed_deleted_files", "feed_audiences",
    "feed_list_of_mentions", "feed_list_of_topics",
    "feed_shared_entity_id", "feed_variant",
]

by_type = defaultdict(list)
for row in feeds:
    by_type[str(row.get("feed_type"))].append(row)

for t, rows in sorted(by_type.items()):
    print(f"\nTYPE={t}  COUNT={len(rows)}")
    for field in fields:
        nn = sum(1 for r in rows if r.get(field) not in (None, "", [], {}))
        if nn:
            print(f"  {field}: {nn}/{len(rows)} ({nn/len(rows):.1%})")


TYPE=post  COUNT=1601
  id: 1601/1601 (100.0%)
  createdbyid: 1601/1601 (100.0%)
  modifiedbyid: 721/1601 (45.0%)
  feed_author_name: 1487/1601 (92.9%)
  feed_data: 1601/1601 (100.0%)
  feed_html_data: 1597/1601 (99.8%)
  feed_site_id: 1406/1601 (87.8%)
  feed_site_name: 1406/1601 (87.8%)
  feed_site_type: 1406/1601 (87.8%)
  feed_content_id: 1257/1601 (78.5%)
  feed_content_type: 1257/1601 (78.5%)
  feed_ref_data: 1595/1601 (99.6%)
  feed_files: 199/1601 (12.4%)
  feed_audiences: 1596/1601 (99.7%)
  feed_list_of_mentions: 781/1601 (48.8%)
  feed_list_of_topics: 28/1601 (1.7%)
  feed_variant: 1601/1601 (100.0%)

TYPE=recognition  COUNT=2179
  id: 2179/2179 (100.0%)
  createdbyid: 2179/2179 (100.0%)
  modifiedbyid: 945/2179 (43.4%)
  feed_author_name: 1907/2179 (87.5%)
  feed_data: 3/2179 (0.1%)
  feed_html_data: 3/2179 (0.1%)
  feed_site_id: 41/2179 (1.9%)
  feed_site_name: 41/2179 (1.9%)
  feed_site_type: 41/2179 (1.9%)
  feed_ref_data: 2/2179 (0.1%)
  feed_associated_entity: 2179/21

### 1.3 FK overlap with existing entities
Check which feed FK fields point to existing People / Site / Content IDs.

In [7]:
entity_sets = {"People": people_ids, "Site": site_ids, "Content": content_ids}

candidates = defaultdict(set)
for row in feeds:
    for field in ["createdbyid", "modifiedbyid", "feed_site_id", "feed_content_id"]:
        v = row.get(field)
        if isinstance(v, str) and v:
            candidates[field].add(v)

    for mention in row.get("feed_list_of_mentions") or []:
        if isinstance(mention, dict):
            mid = mention.get("id")
            if isinstance(mid, str) and mid:
                candidates["mentions[].id"].add(mid)

    fae = row.get("feed_associated_entity")
    if isinstance(fae, dict):
        eid = fae.get("entityId")
        if isinstance(eid, str) and eid:
            candidates["associated_entity.entityId"].add(eid)
        for rr in fae.get("recognitionAwardedToIds") or []:
            if isinstance(rr, dict):
                uid = rr.get("userId")
                if isinstance(uid, str) and uid:
                    candidates["recipients[].userId"].add(uid)

print("Relationship candidate overlaps (scoped):\n")
for field, values in sorted(candidates.items()):
    hits = []
    for label, ids in entity_sets.items():
        overlap = values & ids
        if overlap:
            hits.append(f"{label}:{len(overlap)}")
    hit_str = ", ".join(hits) if hits else "no overlap"
    print(f"  {field:<35} → {hit_str}  (unique={len(values)})")

Relationship candidate overlaps (scoped):

  associated_entity.entityId          → no overlap  (unique=2173)
  createdbyid                         → People:377  (unique=377)
  feed_content_id                     → Content:412  (unique=412)
  feed_site_id                        → Site:29  (unique=29)
  mentions[].id                       → People:316, Site:12  (unique=328)
  modifiedbyid                        → People:275  (unique=275)
  recipients[].userId                 → People:325  (unique=325)


### 1.4 Nested field structure samples

In [8]:
for field in ["feed_associated_entity", "feed_ref_data", "feed_files",
              "feed_list_of_mentions", "feed_audiences"]:
    vals = [r.get(field) for r in feeds if r.get(field) not in (None, "", [], {})]
    print(f"\n{field}: non_empty={len(vals)}")
    if not vals:
        continue
    v = vals[0]
    if isinstance(v, dict):
        print(f"  sample_type=dict  keys={sorted(v.keys())[:12]}")
    elif isinstance(v, list):
        print(f"  sample_type=list  len={len(v)}  item0_type={type(v[0]).__name__ if v else None}")
        if v and isinstance(v[0], dict):
            print(f"  item0_keys={sorted(v[0].keys())[:12]}")
    else:
        print(f"  sample_type={type(v).__name__}  sample={str(v)[:180]}")


feed_associated_entity: non_empty=2179
  sample_type=dict  keys=['entityId', 'metadata', 'recognitionAwardedToIds']

feed_ref_data: non_empty=1597
  sample_type=str  sample=how to get 2024 global holiday list?

feed_files: non_empty=199
  sample_type=list  len=3  item0_type=dict
  item0_keys=['file_additional_attr', 'file_alt_text', 'file_extension', 'file_id', 'file_is_accessible', 'file_is_image', 'file_is_ios_downloadable', 'file_is_video', 'file_mime_type', 'file_name', 'file_provider', 'file_provider_id']

feed_list_of_mentions: non_empty=785
  sample_type=list  len=1  item0_type=dict
  item0_keys=['id', 'name', 'type']

feed_audiences: non_empty=3775
  sample_type=list  len=1  item0_type=str


### 1.5 Exploration summary

**Scope decisions:**
- `timeline` feeds excluded — system-generated, low semantic value
- `post` variants `shared_content` / `shared_recognition` excluded — deferred
- Kept: `post|standard` and `recognition` (all variants)

**Modeling decisions:**
- `Post` and `Recognition` are separate node labels
- `feed_data`, `feed_html_data`, `feed_author_image_url` dropped — bulk text / image URLs, not useful as graph properties
- `feed_site_type`, `feed_content_type` dropped — redundant with the target Site/Content node properties
- `mentions` kept for posts only (recognition recipients serve a similar role)
- `RECOGNIZES` is one-way: `(Recognition)→(People)` — no reverse edge needed
- `associated_entity` not ingested as a relationship — redundant with existing edges

---
## 2 · Extract & Normalise

Call the extractor from `src/`. Same function used by the headless pipeline script.

In [9]:
from src.extractors.feed_extractor import extract_feeds, write_normalized

result = extract_feeds(RAW_FILE)
stats = result["stats"]

print(f"Raw feed rows              : {stats['raw_feed_rows']}")
print(f"Rows kept in scope         : {stats['kept_rows']}")
print(f"Excluded timeline          : {stats['excluded_timeline']}")
print(f"Excluded post non-standard : {stats['excluded_post_non_standard']}")
print(f"Excluded other             : {stats['excluded_other']}")
print(f"Posts extracted             : {stats['posts']}")
print(f"Recognitions extracted     : {stats['recognitions']}")

Raw feed rows              : 6029
Rows kept in scope         : 3780
Excluded timeline          : 2132
Excluded post non-standard : 117
Excluded other             : 0
Posts extracted             : 1601
Recognitions extracted     : 2179


In [10]:
# Persist normalised files
write_normalized(result, NORMALIZED_DIR)
print(f"\nNormalised files written to: {NORMALIZED_DIR}")

Written 1601 post records → /home/ubuntu/graph_experiments/data/normalized/posts.jsonl
Written 2179 recognition records → /home/ubuntu/graph_experiments/data/normalized/recognitions.jsonl

Normalised files written to: /home/ubuntu/graph_experiments/data/normalized


---
## 3 · Analyse Extraction Results

Load the written JSONL files back and verify everything looks correct.

In [11]:
# Load normalised outputs
posts_norm = []
with open(NORMALIZED_DIR / "posts.jsonl") as f:
    for line in f:
        posts_norm.append(json.loads(line))

recognitions_norm = []
with open(NORMALIZED_DIR / "recognitions.jsonl") as f:
    for line in f:
        recognitions_norm.append(json.loads(line))

print(f"posts.jsonl rows        : {len(posts_norm)}")
print(f"recognitions.jsonl rows : {len(recognitions_norm)}")

print("\nPost columns:")
print(sorted(posts_norm[0].keys()))
print("\nRecognition columns:")
print(sorted(recognitions_norm[0].keys()))

posts.jsonl rows        : 1601
recognitions.jsonl rows : 2179

Post columns:
['createdbyid', 'createddate', 'feed_author_name', 'feed_content_id', 'feed_content_is_deleted', 'feed_content_title', 'feed_ref_data', 'feed_site_id', 'feed_site_name', 'feed_type', 'feed_variant', 'id', 'is_deleted', 'lastmodifieddate', 'mentions', 'modifiedbyid']

Recognition columns:
['createdbyid', 'createddate', 'feed_author_name', 'feed_type', 'feed_variant', 'id', 'is_deleted', 'lastmodifieddate', 'modifiedbyid', 'recipients']


In [12]:
# FK validation against normalised entity sets
norm_people_ids  = set()
norm_site_ids    = set()
norm_content_ids = set()

for path, target in [
    (NORMALIZED_DIR / "people.jsonl", norm_people_ids),
    (NORMALIZED_DIR / "sites.jsonl", norm_site_ids),
    (NORMALIZED_DIR / "contents.jsonl", norm_content_ids),
]:
    if path.exists():
        with open(path) as f:
            for line in f:
                rec = json.loads(line)
                if rec.get("id"):
                    target.add(rec["id"])

print(f"Normalised People  : {len(norm_people_ids)}")
print(f"Normalised Site    : {len(norm_site_ids)}")
print(f"Normalised Content : {len(norm_content_ids)}")

# -- Post FKs --
post_creator_ids  = set(r["createdbyid"] for r in posts_norm if r.get("createdbyid"))
post_modifier_ids = set(r["modifiedbyid"] for r in posts_norm if r.get("modifiedbyid"))
post_site_ids     = set(r["feed_site_id"] for r in posts_norm if r.get("feed_site_id"))
post_content_ids  = set(r["feed_content_id"] for r in posts_norm if r.get("feed_content_id"))

post_mention_people, post_mention_site, post_mention_content = set(), set(), set()
for p in posts_norm:
    for m in p.get("mentions") or []:
        mt = (m.get("type") or "").lower()
        mid = m.get("id")
        if not mid:
            continue
        if mt == "people":   post_mention_people.add(mid)
        elif mt == "site":   post_mention_site.add(mid)
        elif mt == "content": post_mention_content.add(mid)

print("\n=== Post FK coverage ===")
print(f"createdbyid      : {len(post_creator_ids)} unique, {len(post_creator_ids - norm_people_ids)} missing People")
print(f"modifiedbyid     : {len(post_modifier_ids)} unique, {len(post_modifier_ids - norm_people_ids)} missing People")
print(f"feed_site_id     : {len(post_site_ids)} unique, {len(post_site_ids - norm_site_ids)} missing Site")
print(f"feed_content_id  : {len(post_content_ids)} unique, {len(post_content_ids - norm_content_ids)} missing Content")
print(f"mentions→People  : {len(post_mention_people)} unique, {len(post_mention_people - norm_people_ids)} missing")
print(f"mentions→Site    : {len(post_mention_site)} unique, {len(post_mention_site - norm_site_ids)} missing")
print(f"mentions→Content : {len(post_mention_content)} unique, {len(post_mention_content - norm_content_ids)} missing")

# -- Recognition FKs --
rec_creator_ids  = set(r["createdbyid"] for r in recognitions_norm if r.get("createdbyid"))
rec_modifier_ids = set(r["modifiedbyid"] for r in recognitions_norm if r.get("modifiedbyid"))

rec_recipient_ids = set()
for r in recognitions_norm:
    for rcp in r.get("recipients") or []:
        uid = rcp.get("user_id") if isinstance(rcp, dict) else None
        if uid:
            rec_recipient_ids.add(uid)

print("\n=== Recognition FK coverage ===")
print(f"createdbyid       : {len(rec_creator_ids)} unique, {len(rec_creator_ids - norm_people_ids)} missing People")
print(f"modifiedbyid      : {len(rec_modifier_ids)} unique, {len(rec_modifier_ids - norm_people_ids)} missing People")
print(f"recipients→People : {len(rec_recipient_ids)} unique, {len(rec_recipient_ids - norm_people_ids)} missing")

Normalised People  : 954
Normalised Site    : 45
Normalised Content : 3142

=== Post FK coverage ===
createdbyid      : 229 unique, 0 missing People
modifiedbyid     : 142 unique, 0 missing People
feed_site_id     : 29 unique, 0 missing Site
feed_content_id  : 412 unique, 0 missing Content
mentions→People  : 312 unique, 0 missing
mentions→Site    : 12 unique, 0 missing
mentions→Content : 0 unique, 0 missing

=== Recognition FK coverage ===
createdbyid       : 329 unique, 0 missing People
modifiedbyid      : 243 unique, 0 missing People
recipients→People : 325 unique, 0 missing


---
## 4 · Ingest into Neo4j

In [18]:
from src.loaders.feed_loader import load_feed_graph
from src.utils.neo4j_client import Neo4jClient

client = Neo4jClient(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
load_feed_graph(client, NORMALIZED_DIR, batch=True)
print("Feed ingestion complete.")

✅ Constraints ensured.
✅ Post nodes loaded: 1601
✅ Recognition nodes loaded: 2179


✅ CREATED (People→Post): 1601
✅ LAST_MODIFIED (People→Post): 721
✅ CREATED (People→Recognition): 2179
✅ LAST_MODIFIED (People→Recognition): 945
✅ BELONGS_TO_SITE (Post→Site): 1406
✅ BELONGS_TO_CONTENT (Post→Content): 1257
✅ RECOGNIZES (Recognition→People): 935
✅ MENTIONS (Post→People|Site|Content): 1531

✅ Feed graph fully loaded.
Feed ingestion complete.


---
## 5 · Post-Ingestion Verification

In [19]:
# Node counts
node_queries = [
    ("Post",        "MATCH (n:Post) RETURN count(n) AS cnt"),
    ("Recognition", "MATCH (n:Recognition) RETURN count(n) AS cnt"),
]

print("=== Node Counts ===")
for label, q in node_queries:
    result = client.query(q)
    print(f"  {label:20s}: {result[0]['cnt']}")

=== Node Counts ===
  Post                : 1601
  Recognition         : 2179


In [20]:
# Relationship counts
rel_queries = [
    ("CREATED (Post)",         "MATCH (:People)-[r:CREATED]->(:Post) RETURN count(r) AS cnt"),
    ("CREATED (Recognition)",  "MATCH (:People)-[r:CREATED]->(:Recognition) RETURN count(r) AS cnt"),
    ("LAST_MODIFIED (Post)",        "MATCH (:People)-[r:LAST_MODIFIED]->(:Post) RETURN count(r) AS cnt"),
    ("LAST_MODIFIED (Recog.)",      "MATCH (:People)-[r:LAST_MODIFIED]->(:Recognition) RETURN count(r) AS cnt"),
    ("BELONGS_TO_SITE (Post)",      "MATCH (:Post)-[r:BELONGS_TO_SITE]->(:Site) RETURN count(r) AS cnt"),
    ("BELONGS_TO_CONTENT (Post)",   "MATCH (:Post)-[r:BELONGS_TO_CONTENT]->(:Content) RETURN count(r) AS cnt"),
    ("MENTIONS",               "MATCH (:Post)-[r:MENTIONS]->() RETURN count(r) AS cnt"),
    ("RECOGNIZES",             "MATCH (:Recognition)-[r:RECOGNIZES]->(:People) RETURN count(r) AS cnt"),
]

print("=== Relationship Counts ===")
for label, q in rel_queries:
    result = client.query(q)
    print(f"  {label:30s}: {result[0]['cnt']}")

=== Relationship Counts ===
  CREATED (Post)                : 1601
  CREATED (Recognition)         : 2179
  LAST_MODIFIED (Post)          : 721
  LAST_MODIFIED (Recog.)        : 945
  BELONGS_TO_SITE (Post)        : 1406
  BELONGS_TO_CONTENT (Post)     : 1257
  MENTIONS                      : 1531
  RECOGNIZES                    : 935


In [21]:
# Distribution queries
print("=== Post variant distribution ===")
rows = client.query("""
    MATCH (p:Post)
    RETURN p.feed_variant AS variant, count(*) AS cnt
    ORDER BY cnt DESC
""")
print(pd.DataFrame(rows).to_string(index=False))

print("\n=== Top 10 Post creators ===")
rows = client.query("""
    MATCH (u:People)-[:CREATED]->(p:Post)
    RETURN u.id AS person_id, u.name AS name, count(p) AS posts
    ORDER BY posts DESC LIMIT 10
""")
print(pd.DataFrame(rows).to_string(index=False))

print("\n=== Top 10 Recognition recipients ===")
rows = client.query("""
    MATCH (r:Recognition)-[:RECOGNIZES]->(u:People)
    RETURN u.id AS person_id, u.name AS name, count(r) AS recognitions
    ORDER BY recognitions DESC LIMIT 10
""")
print(pd.DataFrame(rows).to_string(index=False))

print("\n=== Mention target breakdown ===")
rows = client.query("""
    MATCH (p:Post)-[:MENTIONS]->(t)
    RETURN labels(t)[0] AS target_type, count(*) AS cnt
    ORDER BY cnt DESC
""")
print(pd.DataFrame(rows).to_string(index=False))

=== Post variant distribution ===
 variant  cnt
standard 1601

=== Top 10 Post creators ===
                           person_id name  posts
89f6fc97-6447-4215-9918-4ec35eab1a9f None    106
8744dda0-028a-41c8-9c58-5f7c295de819 None     90
3a331b59-add7-4198-a9a8-78c11af00261 None     51
e8bcab12-2978-4c65-a511-8d6d62c27102 None     50
0344baf1-a4f8-48c2-a401-c8d10fbabeac None     49
2b95ec70-7644-4b8b-aa9c-804243324c39 None     49
38f01cba-3a37-4493-847d-0f166849af5c None     48
b9f195f6-e458-4c18-96d3-0d7f14962a08 None     46
fa4f7f63-5c35-40a8-ad51-f0f982706f27 None     43
acc88c24-7434-47d7-bd00-69b0890b9af8 None     39

=== Top 10 Recognition recipients ===
                           person_id name  recognitions
b9f195f6-e458-4c18-96d3-0d7f14962a08 None            19
42b49537-a9dc-492e-9103-d1c9765ff5df None            16
b5f00f33-0286-42ae-a926-86bee5e20cde None            13
7113b4ab-ddc1-4e8b-b73c-564ae05bc087 None            13
44f3a747-706d-4189-b3a4-bbfdfa3d9bd5 None         

In [22]:
client.close()
print("Neo4j connection closed.")

Neo4j connection closed.
